In [ ]:
import pandas as pd
import pymysql
from pymongo import MongoClient
import psycopg2
import matplotlib.pyplot as plt
import time as tm
import seaborn as sns
%matplotlib inline
import json
import urllib.request
from math import radians, cos, sin, asin, sqrt
import matplotlib
import numpy as np
import datetime
from datetime import datetime

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py

db = client.get_database('telematics_db')




def gettime(x):
    try:
        return pd.Timestamp(x).timestamp()
    except:
        return None

In [ ]:
vehicle_details=pd.read_sql("select sub_type_installation,vehicle_id,model_name,data_source,vehicle_state,live_time,deboarding_time from vehicle_details where live_time<'Wednesday, May 1, 2019 12:00 PM'",con=postgres)
print('vehicle_details ho gyaa')
device_models=pd.read_sql("select * from device_models",con=mysql_cn)

our_devices=['GT06N','WETRACK2','WETRACK-LITE','C100','TL1000','TL250','WETRACK800','AW100','WP30L','GRLAIS','AT50','WP30L','FMB920','FMB207','FMB150','FMB001','LC100_SOS','WETRACK2_SOS','AT4','ATLANTA_OBD','BHARAT101','TRANSYNC_AIS_140','GAGAN']                           


our_device_list=[]

for i, row in device_models.iterrows():
    if row['name'] in our_devices:
        our_device_list.append('yes')
    else:
        our_device_list.append('no')
device_models['our_device']=our_device_list

In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    #print(lon1,lat1,lon2,lat2)
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

In [ ]:
def average_delay(vid_list):
    final=pd.DataFrame()
    sortOrder='desc'
    timeCondition='lte'
    for  vehicle_id in vid_list:
    
        data=pd.DataFrame()

        print('.')
        for i in range(0,1):
            print('in range ' + str(i))
            from_tm=(1559347200 -(i+1)*86400)
            to_tm=(1559347200-i*86400)
            #global url
            url='http://db.example.internal:8080/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder={}&timeCondition={}'.format(vehicle_id,from_tm,to_tm,'desc','gte')
            data_json=json.load(urllib.request.urlopen(url))

            try:
                reuse=pd.DataFrame(data_json['data'][str(vehicle_id)])        
                data=pd.concat([data,reuse])
                #print('day' + str(i))
                #print(data)

            except:
                print('no_DATA')
        #global device_history
        #print(data)
        try:
            data.drop_duplicates('time',inplace=True)
        except:
            print("no_data")

        device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':int(vehicle_id),'fromTime':{'$lte':1559347200,'$gte':1559347200-86400}})))
        try:
            device_history=device_history[device_history['toTime'].notnull()]
            for i,row in device_history.iterrows():
                data=data[data['time']<row['fromTime']][data['time']>row['toTime']]
        
        except:
            print('no_no_info')
        
        #print(data)
        
        if len(data)>10:
            

            data['created_epoch']=data['created']/1000
            data['avg_delay']=data['created_epoch']-data['time']
            data['loc2']=data.location.shift(1)
            data = data.iloc[1:]
            dis=[]
            for l,lrow in data.iterrows():
                dis.append(haversine(lrow['location'],lrow['loc2']))
            data['distance']=dis

            #data['distance']=data.apply(lambda row : haversine(row["location"],row["loc2"]), axis=1)

            median_val=data['avg_delay'].median()
            mean_val=data['avg_delay'].mean()

            total_dist=data.distance.sum()
            jump_dist=data[data['distance']>500]['distance'].sum()
            total_ping_instances=len(data)
            total_jump_ping_instances=len(data[data['distance']>500])

            frame=pd.DataFrame([[vehicle_id,
                                 median_val,
                                 mean_val,
                                 total_dist,
                                 jump_dist,
                                total_ping_instances,
                                total_jump_ping_instances]],
                               columns=['vehicle_id',
                                        'avg_delay_month_median',
                                        'avg_delay_month_mean',
                                        'total_dist',
                                       'jump_dist',
                                       'total_ping_instances',
                                       'total_jump_ping_instances'])
            print(frame)
            #print(frame)

            final=pd.concat([final,frame])
        else:
            print('no_info khaa gaya')
       # print(final)
    return final






    

In [ ]:
veh=pd.read_sql("select * from operator_vehicles where deboarding_time is null",con=mysql_cn)
vehicle_list=veh.vehicle_id.unique()

In [ ]:

df_split = np.array_split(vehicle_list, 10)



import concurrent.futures

pool = concurrent.futures.ThreadPoolExecutor(max_workers = 10)
lengths = pool.map(average_delay, df_split)


In [ ]:
aa_gaya=pd.concat(lengths)

In [ ]:
aa_gaya.to_pickle("avg_delay_pickel.pkl")

In [ ]:
datetime.fromtimestamp(87273681)

In [ ]:
from tqdm import tqdm
import time as time

In [ ]:
for i in tqdm(range(100)):
    time.sleep(1)
    a=2
    